# 02 — Evaluate PERPHECT Model

Load a trained model and the held-out test set, then display a full metrics suite:
- Classification report (precision, recall, F1)
- Confusion matrix
- ROC-AUC curve
- Precision-recall curve
- Prediction distribution histogram
- Per-source metrics (private_data vs generated negatives)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

sys.path.insert(0, str(Path.cwd()))
from train import build_model

## 1. Configuration

In [ ]:
# Set these paths to your trained model and test data
MODEL_PATH = Path("/results")  # path to model_best.keras
TEST_DATA = Path.cwd() / "test_data"
BACTERIUM_THRESHOLD = 7_000_000
PHAGE_THRESHOLD = 200_000

## 2. Load Test Data

In [ ]:
csv_path = TEST_DATA / "test_set.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"No test data found at {csv_path}. Re-run 01_prepare_test_set.ipynb.")

test_df = pd.read_csv(csv_path)
print(f"Loaded {len(test_df)} test pairs from {csv_path}")
print(f"\nLabel distribution:")
print(test_df["label"].value_counts().to_string())
print(f"\nSource distribution:")
print(test_df["source"].value_counts().to_string())

## 3. Load Model

In [ ]:
# Find the model file
model_candidates = list(MODEL_PATH.rglob("model_best.keras"))
if not model_candidates:
    model_candidates = list(MODEL_PATH.rglob("model_final.keras"))
if not model_candidates:
    raise FileNotFoundError(f"No .keras model found under {MODEL_PATH}")

model_path = model_candidates[0]
print(f"Loading model from: {model_path}")

import keras
model = keras.models.load_model(model_path)
model.summary()
print(f"\nModel loaded: {model.count_params():,} parameters")

## 4. Generate Predictions

In [ ]:
from pbi import quick_connect
from pbi_adapter import PBIAdapter

retriever = quick_connect()
adapter = PBIAdapter(
    retriever,
    bacterium_threshold=BACTERIUM_THRESHOLD,
    phage_threshold=PHAGE_THRESHOLD,
)

# Fetch sequences and encode for each test pair
print("Fetching sequences from database...")
bacteria_seqs = []
phage_seqs = []
valid_mask = []

for i, row in test_df.iterrows():
    bseq = adapter._fetch_host_sequence(row["Host_ID"])
    pseq = adapter._fetch_phage_sequence(row["Phage_ID"])
    if bseq is not None and pseq is not None:
        bacteria_seqs.append(adapter._pad_and_encode(bseq, BACTERIUM_THRESHOLD))
        phage_seqs.append(adapter._pad_and_encode(pseq, PHAGE_THRESHOLD))
        valid_mask.append(True)
    else:
        valid_mask.append(False)

valid_mask = np.array(valid_mask)
n_valid = valid_mask.sum()
n_dropped = len(valid_mask) - n_valid
if n_dropped > 0:
    print(f"Dropped {n_dropped} pairs (sequences too short)")

bacteria_arr = np.stack(bacteria_seqs)
phage_arr = np.stack(phage_seqs)
labels_valid = test_df["label"].values[valid_mask]
sources_valid = test_df["source"].values[valid_mask]

print(f"Prediction input: bacteria={bacteria_arr.shape}, phage={phage_arr.shape}")

# Run predictions
predictions = model.predict([bacteria_arr, phage_arr], verbose=1).flatten()
pred_labels = (predictions > 0.5).astype(int)

print(f"\nPredictions generated: {len(predictions)}")
print(f"Positive predictions: {pred_labels.sum()}")
print(f"Negative predictions: {len(pred_labels) - pred_labels.sum()}")

## 5. Classification Report

In [ ]:
print(classification_report(
    labels_valid, pred_labels,
    target_names=["Negative", "Positive"],
))

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(labels_valid, pred_labels)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title("Confusion Matrix")
plt.colorbar(im, ax=ax)
tick_marks = np.arange(2)
ax.set_xticks(tick_marks)
ax.set_xticklabels(["Negative", "Positive"])
ax.set_yticks(tick_marks)
ax.set_yticklabels(["Negative", "Positive"])
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

## 7. ROC-AUC Curve

In [ ]:
fpr, tpr, _ = roc_curve(labels_valid, predictions)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="darkorange", lw=2,
        label=f"ROC curve (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
print(f"ROC-AUC: {roc_auc:.4f}")

## 8. Precision-Recall Curve

In [ ]:
precision, recall, _ = precision_recall_curve(labels_valid, predictions)
avg_precision = average_precision_score(labels_valid, predictions)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, color="blue", lw=2,
        label=f"PR curve (AP = {avg_precision:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()
print(f"Average Precision: {avg_precision:.4f}")

## 9. Prediction Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Separate predictions by true label
pos_preds = predictions[labels_valid == 1]
neg_preds = predictions[labels_valid == 0]

ax.hist(neg_preds, bins=50, alpha=0.6, label="True Negative", color="steelblue")
ax.hist(pos_preds, bins=50, alpha=0.6, label="True Positive", color="coral")
ax.axvline(x=0.5, color="black", linestyle="--", label="Threshold (0.5)")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Count")
ax.set_title("Prediction Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Per-Source Metrics

In [ ]:
# Break down metrics by negative source
unique_sources = sorted(set(sources_valid))

for source in unique_sources:
    mask = sources_valid == source
    if mask.sum() == 0:
        continue
    src_labels = labels_valid[mask]
    src_preds = pred_labels[mask]
    src_probs = predictions[mask]

    print(f"\n{'='*50}")
    print(f"Source: {source} ({mask.sum()} pairs)")
    print(f"{'='*50}")
    print(classification_report(
        src_labels, src_preds,
        target_names=["Negative", "Positive"],
        zero_division=0,
    ))

    if len(set(src_labels)) > 1:
        fpr_s, tpr_s, _ = roc_curve(src_labels, src_probs)
        print(f"  ROC-AUC: {auc(fpr_s, tpr_s):.4f}")

## Summary

Key metrics to report:

| Metric | Value |
|--------|-------|
| ROC-AUC | See section 7 |
| Average Precision | See section 8 |
| F1 (Positive) | See section 5 |
| Confusion Matrix | See section 6 |

**Interpretation guide:**
- **ROC-AUC > 0.9**: Excellent discrimination
- **ROC-AUC 0.7-0.9**: Good discrimination
- **ROC-AUC < 0.7**: Model needs improvement
- High precision = few false positives (reliable positive predictions)
- High recall = few false negatives (catches most true positives)